In [ ]:
# # Custom Artificial Neural Network (ANN) with Keras & TensorFlow
#
# This notebook contains the complete construction, optimization, training history analysis, and performance validation of a custom deep **Artificial Neural Network (ANN)** built using Keras and TensorFlow.
#
# ### Mathematical Concepts & Intuition
# Unlike basic Scikit-Learn classifiers, an ANN allows deep structural flexibility. We stack multiple dense connected layers with custom **Dropout regularization layers** to drop random nodes and prevent overfitting:
#
# #### Key Components:
# 1. **Dense Layers (Fully Connected)**: Multiplies inputs by a weight matrix, adds bias, and passes through activation (ReLU).
# 2. **Dropout Layers (Regularization)**: Randomly sets a fraction (e.g., $20\%$) of input units to 0 at each update during training time, promoting weight distribution.
# 3. **Binary Cross-Entropy Loss (Target binary classification)**:
#    $$\text{Loss} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$
# 4. **Adam Optimizer**: An adaptive learning rate optimization algorithm that utilizes first and second moments of gradients.
#
# ### Hyperparameter Configurations Tested
# We evaluate two highly customized deep neural architectures:
# *   **Config 1**: 3-layer deep structure `[128 -> 64 -> 32]` | Activation: ReLU | Learning Rate: `0.001` | Epochs: `20` | Batch Size: `64`
# *   **Config 2**: 2-layer wide structure `[256 -> 128]` | Activation: ReLU | Learning Rate: `0.0005` | Epochs: `20` | Batch Size: `128`


In [ ]:
# Step 1: Imports and environment setup
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)


In [ ]:
# ### Step 2: Load Preprocessed Features
# Load processed scaling vectors from `../data/processed_data.joblib`.


In [ ]:
# Step 2: Import data
PROCESSED_DATA_PATH = "../data/processed_data.joblib"
if not os.path.exists(PROCESSED_DATA_PATH):
    raise FileNotFoundError(f"Preprocessed data not found at {PROCESSED_DATA_PATH}")

data = joblib.load(PROCESSED_DATA_PATH)
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
print(f"Input Feature Dimensions: {X_train.shape[1]}")


In [ ]:
# ### Step 3: Model Architecture Creation & Parameter Search
# We construct the models using Keras `Sequential` API and optimize validation split values.


In [ ]:
# Step 3: Model Building and Training Loops
input_dim = X_train.shape[1]

# What is a Hidden Layer, Dense, and Dropout?
# - Dense Layer: A standard fully-connected neural network layer where every neuron receives inputs from all neurons in the previous layer.
# - Dropout Layer: A regularizer that randomly turns off a fraction (e.g. 20%) of neurons during each training step. This prevents the model from relying too much on specific single weather factors (like absolute temperature) and forces it to distribute weight across multiple factors, preventing overfitting.
def create_ann(layers_config, activation='relu', lr=0.001, dropout=0.2):
    model = Sequential()
    # First Hidden Layer: Takes raw features as inputs
    model.add(Dense(layers_config[0], input_shape=(input_dim,), activation=activation))
    model.add(Dropout(dropout))
    
    # Additional Deep Hidden Layers: Learns high-level abstractions
    for neurons in layers_config[1:]:
        model.add(Dense(neurons, activation=activation))
        model.add(Dropout(dropout))
        
    # Output Layer: Sigmoid activation outputs a probability between 0 and 1 (rain probability)
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile model
    model.compile(optimizer=Adam(learning_rate=lr), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

# Deep Network Configuration Settings:
# - Config 1: Deep & Narrow [128 -> 64 -> 32] layers. High capacity for deep sequential abstractions, learning rate 0.001.
# - Config 2: Shallow & Wide [256 -> 128] layers. Large capacity in earlier layers, learning rate 0.0005.
configs = [
    {'layers': [128, 64, 32], 'activation': 'relu', 'lr': 0.001, 'epochs': 20, 'batch_size': 64},
    {'layers': [256, 128], 'activation': 'relu', 'lr': 0.0005, 'epochs': 20, 'batch_size': 128}
]

results = []
best_acc = 0
best_model = None
best_history = None

print("--- Starting Custom ANN Training and Comparison ---")
for idx, config in enumerate(configs):
    print(f"\nTraining Configuration {idx+1}: layers={config['layers']} | lr={config['lr']}")
    # Build ANN
    model = create_ann(config['layers'], config['activation'], config['lr'])
    
    # Fit with 20% validation split
    history = model.fit(X_train, y_train, 
                        epochs=config['epochs'], 
                        batch_size=config['batch_size'], 
                        verbose=1, 
                        validation_split=0.2)
    
    # Evaluate testing accuracy
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, y_pred)
    print(f"  --> Configuration {idx+1} Test Accuracy: {acc * 100:.2f}%")
    
    results.append({
        'config': config,
        'accuracy': acc,
        'history': history.history
    })
    
    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_history = history.history

print("\nCustom ANN Training Complete!")


In [ ]:
# ### Step 4: Metric Evaluation & Loss Optimization Charts
# We render the Confusion Matrix and compare Training Loss vs. Validation Loss curves.


In [ ]:
# Step 4: Evaluation and Plotting
y_pred_prob = best_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=== BEST ANN CONFIGURATION METRICS ===")
print(f"Optimal Test Accuracy:     {accuracy * 100:.2f}%")
print(f"Precision Score:           {precision * 100:.2f}%")
print(f"Recall Score:              {recall * 100:.2f}%")
print(f"F1 Performance:            {f1 * 100:.2f}%")

# Set up side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Plot Heatmap Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['No Rain', 'Rain'], yticklabels=['No Rain', 'Rain'], ax=axes[0])
axes[0].set_title('Confusion Matrix - Custom ANN (Best Config)', fontsize=13, pad=10)
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# 2. Plot Keras Validation Loss History
axes[1].plot(best_history['loss'], label='Training Loss', color='purple', linewidth=2)
axes[1].plot(best_history['val_loss'], label='Validation Loss', color='darkviolet', linewidth=2, linestyle='--')
axes[1].set_title('ANN Optimization History (Loss Curves)', fontsize=13, pad=10)
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss (Binary Cross-Entropy)')
axes[1].legend()
axes[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()


In [ ]:
# ### Step 5: Save Custom ANN Model
# Save model architecture and weights in standard Keras format to `../data/models/ann.keras`.


In [ ]:
# Step 5: Export optimal ANN
MODELS_DIR = "../data/models"
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

model_path = os.path.join(MODELS_DIR, 'ann.keras')
best_model.save(model_path)
print(f"Optimal ANN Keras model successfully saved to: {model_path}")
